In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


# Pilot — E0: fixed weights, pure PINN, E1A and E3A, seeds 0-2

Experiments v01, E0. Full J-V curve per run (Algorithm A2): equilibrium, then one
voltage after another until $J$ changes sign, Adam then L-BFGS per voltage, validation
$e_{PCE}$ at the iteration marks 1000-5000 for the budget rule.

**Rehearsal by default.** `PINN_DPV_REHEARSAL` unset or `1` runs every configuration
shrunk (`steps.REHEARSAL`) under run ids prefixed `rehearsal-`: the Kaggle GPU check.
Only `PINN_DPV_REHEARSAL=0` runs E0 at full scale — the Colab A100, launched by the
operator.

**Nothing is lost between sessions.** Each run checkpoints atomically every 500
iterations and at the end of every voltage; light results are pushed to the `results`
branch of the repository and the checkpoint to the run's release when a `GH_TOKEN`
secret is present, and the run folder is mirrored to `PINN_DPV_MIRROR` (a Drive folder
on Colab). Re-running this notebook resumes every unfinished run from the newest copy.

### On Colab (A100)

Do not open this notebook directly: open `colab/run_all.ipynb`, which clones the
repository, downloads the dataset from Kaggle, mounts Drive and runs sigma, pilot and
report in order, with progress. `GH_TOKEN` goes in Colab's Secrets panel (fine-grained,
contents write on this repository only) — never in a cell.

In [ ]:
import sys

sys.path.insert(0, str(ROOT / "src"))
# The fingerprint of the code this notebook ran against; `verify` recomputes it.
from PINN_DPV import report_digest

print(report_digest.stamp())


In [ ]:
import os

import jax

from PINN_DPV import data, steps

REHEARSAL = os.environ.get("PINN_DPV_REHEARSAL", "1") != "0"
print("devices:", jax.devices())
print("data:", data.dataset_root(), "| mirror:", os.environ.get("PINN_DPV_MIRROR"), "| rehearsal:", REHEARSAL)
missing = [p for fs in steps.PILOT_FAMILY_STAGES for p in data.canonical_paths(fs) if not Path(p).exists()]
assert not missing, f"dPV files missing: {missing}"
assert all(Path(steps.SIGMA_DIR, f"{fs}_seed{s}.json").exists()
           for fs in steps.PILOT_FAMILY_STAGES for s in steps.PILOT_SEEDS), "run the sigma step first"

In [ ]:
from PINN_DPV import persistence

PUSH = persistence.github_token() is not None
print("push to GitHub:", PUSH)
summaries = steps.run_pilot(rehearsal=REHEARSAL, push=PUSH)

## Per run

In [ ]:
for s in summaries:
    marks = s.get("mean_e_pce_val_by_mark") or {}
    print(f"{s['run_id']:<48} {s['seconds']:9.0f} s  val {s.get('mean_e_pce_val', float('nan')):8.4f} pp  "
          f"test {s.get('mean_e_pce_test', float('nan')):8.4f} pp  marks " +
          " ".join(f"{k}:{v:.3f}" for k, v in marks.items()))